In [ ]:
from pathlib import Path
import pickle

import mne
import numpy as np

from scipy.signal import spectrogram


import sys
import os

In [ ]:
from pathlib import Path

current_dir = Path.cwd()

if "ESPECT_CONV" in current_dir.parts:
    idx = current_dir.parts.index("ESPECT_CONV")
    PROJECT_ROOT = Path(*current_dir.parts[:idx + 1])
else:
    PROJECT_ROOT = current_dir

INPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_30_array_assembly"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_31_array_images"

EPOCH_START_SECONDS = 1.0
EPOCH_END_SECONDS = 3.5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Raiz do Projeto:", PROJECT_ROOT)
print("Entrada:", INPUT_DIR)
print("Saída:", OUTPUT_DIR)

In [ ]:
epoch_files = sorted(INPUT_DIR.glob("*_power.npy"))

print(f"Sessões encontradas: {len(epoch_files)}")
for path in epoch_files[:6]:
    print(" -", path.relative_to(PROJECT_ROOT))

In [ ]:
data_path = epoch_files[0]

power_array = np.load(data_path)
print(power_array.shape)

frequencies_path = INPUT_DIR/"frequencies.npy"
frequencies = np.load(frequencies_path)

times_path = INPUT_DIR/"times.npy"
times = np.load(times_path)

epoch = 5
time_window = 3

cube = power_array[
    epoch,
    :,
    :,
    :,
    time_window
]

print(cube.shape)

In [ ]:
import numpy as np
import pyvista as pv

print("Formato do cubo:", cube.shape)
print("Menor potência:", cube.min())
print("Maior potência:", cube.max())

# Configura a visualização interativa no Jupyter.
pv.set_jupyter_backend("client")

# Garante que os dados são float.
volume = cube.astype(np.float32)

# Cria a grade 3D.
grid = pv.ImageData(
    dimensions=volume.shape
)

# Define o espaçamento físico entre os pontos nos três eixos:
# row, column e frequency.
grid.spacing = (
    1.0,
    1.0,
    1.0
)

# Define a origem do volume.
grid.origin = (
    0.0,
    0.0,
    0.0
)

# Coloca os valores de potência nos pontos da grade.
#
# order="F" é importante porque o VTK/PyVista organiza os
# valores internos seguindo a ordem Fortran.
grid.point_data["power"] = volume.ravel(order="F")

# Cria o visualizador.
plotter = pv.Plotter(notebook=True)

# Adiciona o volume.
plotter.add_volume(
    grid,
    scalars="power",
    cmap="hot",
    opacity="sigmoid",
    shade=False
)

plotter.add_axes()

plotter.add_text(
    (
        f"Época: {epoch}\n"
        f"Janela temporal: {time_window}\n"
        f"Centro da janela: {times[time_window]:.3f} s"
    ),
    font_size=10
)

plotter.show(
    jupyter_backend="client"
)